In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
import kagglehub

# Download latest version 
# Check version: python --version
# Install Python: python -m pip install numpy pandas tensorflow scikit-learn kagglehub
# Code: https://www.kaggle.com/datasets/burnoutminer/heights-and-weights-dataset/data
path = kagglehub.dataset_download("burnoutminer/heights-and-weights-dataset")

print("Path to dataset files:", path)

c:\Users\keely\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\keely\.cache\kagglehub\datasets\burnoutminer\heights-and-weights-dataset\versions\1


In [3]:
for file in os.listdir(path):
    print(file)

SOCR-HeightWeight.csv


In [4]:
df=pd.read_csv(os.path.join(path, "SOCR-HeightWeight.csv" ))
print(df.head())
print(df.columns)

   Index  Height(Inches)  Weight(Pounds)
0      1        65.78331        112.9925
1      2        71.51521        136.4873
2      3        69.39874        153.0269
3      4        68.21660        142.3354
4      5        67.78781        144.2971
Index(['Index', 'Height(Inches)', 'Weight(Pounds)'], dtype='str')


In [6]:
df["Height_cm"] = df["Height(Inches)"] * 2.54
df["Weight_kg"] = df["Weight(Pounds)"] * 0.45359237

In [7]:
data=df[["Height_cm","Weight_kg"]]
print(data.head())

    Height_cm  Weight_kg
0  167.089607  51.252536
1  181.648633  61.909598
2  176.272800  69.411834
3  173.270164  64.562251
4  172.181037  65.452064


In [9]:
x = data[["Height_cm"]].values.reshape(-1,1)
y = data["Weight_kg"].values.reshape(-1,1)

In [11]:
scaler_X=StandardScaler()
scaler_Y=StandardScaler()
X_scaled=scaler_X.fit_transform(x)
Y_scaled=scaler_Y.fit_transform(y)

In [ ]:
base = tf.keras.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(1)
])

In [13]:
base.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=0.01), loss='mse')
base.fit(X_scaled, Y_scaled, epochs=200, verbose=0)

In [14]:
X_mean, X_std = float(scaler_X.mean_[0]), float(np.sqrt(scaler_X.var_[0]))
Y_mean, Y_std = float(scaler_Y.mean_[0]), float(np.sqrt(scaler_Y.var_[0]))

In [17]:
inp = layers.Input(shape=(1,), name="Height_cm")
x = layers.Lambda(lambda t:(t-X_mean)/X_std, name="standardize_x")(inp)
z = layers.Dense(1, name="linear")(x)
out=layers.Lambda(lambda t:t*Y_std+Y_mean,name="destandardize_y")(z)

model2=models.Model(inp,out)

In [18]:
model2.get_layer("linear").set_weights(base.layers[0].get_weights())


In [19]:
for h in [150,170,180,190,0]:
    pred = model2.predict(np.array([[h]], dtype=np.float32), verbose=0)[0,0]
    print(f"Height: {h:.1f} cm => Weight: {pred:.2f} kg")


Height: 150.0 cm => Weight: 45.25 kg
Height: 170.0 cm => Weight: 56.09 kg
Height: 180.0 cm => Weight: 61.51 kg
Height: 190.0 cm => Weight: 66.93 kg
Height: 0.0 cm => Weight: -36.05 kg


In [21]:
convert = tf.lite.TFLiteConverter.from_keras_model(model2)
tflite_model = convert.convert()
with open("height_weight.tflite","wb") as f:
    f.write(tflite_model)
    print("success model has been saved")



INFO:tensorflow:Assets written to: C:\Users\keely\AppData\Local\Temp\tmp_34ngoyb\assets


INFO:tensorflow:Assets written to: C:\Users\keely\AppData\Local\Temp\tmp_34ngoyb\assets


Saved artifact at 'C:\Users\keely\AppData\Local\Temp\tmp_34ngoyb'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 1), dtype=tf.float32, name='Height_cm')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2010357925072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2010357926032: TensorSpec(shape=(), dtype=tf.resource, name=None)
success model has been saved
